In [ ]:
## IMPORTS 
import os
import sys 
current_dir = os.getcwd()
LOCSCALE_2_SCRIPTS_PATH = os.path.dirname(current_dir)
sys.path.append(LOCSCALE_2_SCRIPTS_PATH)
from scripts.utils.plot_utils import *

import numpy as np
import random
import matplotlib.pyplot as plt
from matplotlib.pyplot import cm
from scipy import stats
import pandas as pd
import seaborn as sns
from scripts.utils.plot_utils import plot_correlations
from scripts.utils.general import setup_environment, create_folders_if_they_do_not_exist


random.seed(42)
np.random.seed(42)

PLOT_DATA_STORE_PATH = setup_environment(current_dir)

# Figure 6 h 

In [ ]:
recall_data_0665_save_path = os.path.join(PLOT_DATA_STORE_PATH, "recall_data_0665.csv")
recall_data_0665 = pd.read_csv(recall_data_0665_save_path)

In [ ]:
palette = {"Feature Enhanced": "#56b4e9", 
              "DeepEMhancer": "#e69f00", 
              "EMready": "#aa55ff"}
sns.lineplot(data=recall_data_0665, x='Threshold', y='Recall', hue='Method', palette=palette)

## Figure 6 i

In [ ]:
aurc_data_save_path = os.path.join(PLOT_DATA_STORE_PATH, "aurc_data_all.csv")
aurc_data_long = pd.read_csv(aurc_data_save_path)



In [ ]:
sns.boxplot(data=aurc_data_long, x='Method', y='AURC', color="white", showfliers=False)
sns.swarmplot(data=aurc_data_long, x='Method', y='AURC', hue='Method', palette=palette, dodge=False, size=6)


In [ ]:
# run permutation tests to compute the significance of the differences between the methods
from scipy.stats import permutation_test

# test between feature enhanced and deepemhancer
def run_permutation_test(data1, data2, num_permutations=10000):
    def statistic(x, y):
        return np.mean(x) - np.mean(y)

    result = permutation_test((data1, data2), statistic, n_resamples=num_permutations, alternative='two-sided')
    # print 
    #print(f"pvalue: {result.pvalue}, statistic: {result.statistic} significance: {result.pvalue < 0.05}")
    return result

results = {}

fe_area = aurc_data_long[aurc_data_long["Method"] == "Feature Enhanced"]["AURC"]
deepemhancer_area = aurc_data_long[aurc_data_long["Method"] == "DeepEMhancer"]["AURC"]
emready_area = aurc_data_long[aurc_data_long["Method"] == "EMready"]["AURC"]


fe_deepemhancer = run_permutation_test(fe_area, deepemhancer_area)
fe_emready = run_permutation_test(fe_area, emready_area)
deepEMhancer_EMready = run_permutation_test(deepemhancer_area, emready_area)

print("Feature Enhanced vs DeepEMhancer", fe_deepemhancer.pvalue)
print("Feature Enhanced vs EMready", fe_emready.pvalue)
print("DeepEMhancer vs EMready", deepEMhancer_EMready.pvalue)

